In [ ]:
import random
import os
import timeit
import copy

import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

# logistic function
from scipy.special import expit

#SKlearn imports
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier as Sklearn_GBC
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.decomposition import PCA

In [ ]:
class BaseEstimator:
    y_required = True
    fit_required = True

    def _setup_input(self, X, y=None):
        """Ensure inputs to an estimator are in the expected format.

        Ensures X and y are stored as numpy ndarrays by converting from an
        array-like object if necessary. Enables estimators to define whether
        they require a set of y target values or not with y_required, e.g.
        kmeans clustering requires no target labels and is fit against only X.

        Parameters
        ----------
        X : array-like
            Feature dataset.
        y : array-like
            Target values. By default is required, but if y_required = false
            then may be omitted.
        """
        if not isinstance(X, np.ndarray):
            X = np.array(X)

        if X.size == 0:
            raise ValueError("Got an empty matrix.")

        if X.ndim == 1:
            self.n_samples, self.n_features = 1, X.shape
        else:
            self.n_samples, self.n_features = X.shape[0], np.prod(X.shape[1:])

        self.X = X

        if self.y_required:
            if y is None:
                raise ValueError("Missed required argument y")

            if not isinstance(y, np.ndarray):
                y = np.array(y)

            if y.size == 0:
                raise ValueError("The targets array must be no-empty.")

        self.y = y

    def fit(self, X, y=None):
        self._setup_input(X, y)

    def predict(self, X=None):
        if not isinstance(X, np.ndarray):
            X = np.array(X)

        if self.X is not None or not self.fit_required:
            return self._predict(X)
        else:
            raise ValueError("You must call `fit` before `predict`")

    def _predict(self, X=None):
        raise NotImplementedError()

In [ ]:
def f_entropy(p):
    # Convert values to probability
    p = np.bincount(p) / float(p.shape[0])

    ep = stats.entropy(p)
    if ep == -float("inf"):
        return 0.0
    return ep


def information_gain(y, splits):
    splits_entropy = sum(
        [f_entropy(split) * (float(split.shape[0]) / y.shape[0]) for split in splits]
    )
    return f_entropy(y) - splits_entropy


def mse_criterion(y, splits):
    y_mean = np.mean(y)
    return -sum(
        [
            np.sum((split - y_mean) ** 2) * (float(split.shape[0]) / y.shape[0])
            for split in splits
        ]
    )


def xgb_criterion(y, left, right, loss):
    left = loss.gain(left["actual"], left["y_pred"])
    right = loss.gain(right["actual"], right["y_pred"])
    initial = loss.gain(y["actual"], y["y_pred"])
    gain = left + right - initial
    return gain


def get_split_mask(X, column, value):
    left_mask = X[:, column] < value
    right_mask = X[:, column] >= value
    return left_mask, right_mask


def split(X, y, value):
    left_mask = X < value
    right_mask = X >= value
    return y[left_mask], y[right_mask]


def split_dataset(X, target, column, value, return_X=True):
    left_mask, right_mask = get_split_mask(X, column, value)

    left, right = {}, {}
    for key in target.keys():
        left[key] = target[key][left_mask]
        right[key] = target[key][right_mask]

    if return_X:
        left_X, right_X = X[left_mask], X[right_mask]
        return left_X, right_X, left, right
    else:
        return left, right


In [ ]:
random.seed(111)

In [ ]:
class Tree(object):
    """Recursive implementation of decision tree."""

    def __init__(self, regression=False, criterion=None, n_classes=None):
        self.regression = regression
        self.impurity = None
        self.threshold = None
        self.column_index = None
        self.outcome = None
        self.criterion = criterion
        self.loss = None
        self.n_classes = n_classes  # Only for classification

        self.left_child = None
        self.right_child = None

    @property
    def is_terminal(self):
        return not bool(self.left_child and self.right_child)

    def _find_splits(self, X):
        """Find all possible split values."""
        split_values = set()

        # Get unique values in a sorted order
        x_unique = list(np.unique(X))
        for i in range(1, len(x_unique)):
            # Find a point between two values
            average = (x_unique[i - 1] + x_unique[i]) / 2.0
            split_values.add(average)

        return list(split_values)

    def _find_best_split(self, X, target, n_features):
        """Find best feature and value for a split. Greedy algorithm."""

        # Sample random subset of features
        subset = random.sample(list(range(0, X.shape[1])), n_features)
        max_gain, max_col, max_val = None, None, None

        for column in subset:
            split_values = self._find_splits(X[:, column])
            for value in split_values:
                if self.loss is None:
                    # Random forest
                    splits = split(X[:, column], target["y"], value)
                    gain = self.criterion(target["y"], splits)
                else:
                    # Gradient boosting
                    left, right = split_dataset(
                        X, target, column, value, return_X=False
                    )
                    gain = xgb_criterion(target, left, right, self.loss)

                if (max_gain is None) or (gain > max_gain):
                    max_col, max_val, max_gain = column, value, gain
        return max_col, max_val, max_gain

    def _train(
        self,
        X,
        target,
        max_features=None,
        min_samples_split=10,
        max_depth=None,
        minimum_gain=0.01,
    ):
        try:
            # Exit from recursion using assert syntax
            assert X.shape[0] > min_samples_split
            assert max_depth > 0

            if max_features is None:
                max_features = X.shape[1]

            column, value, gain = self._find_best_split(X, target, max_features)
            assert gain is not None
            if self.regression:
                assert gain != 0
            else:
                assert gain > minimum_gain

            self.column_index = column
            self.threshold = value
            self.impurity = gain

            # Split dataset
            left_X, right_X, left_target, right_target = split_dataset(
                X, target, column, value
            )

            # Grow left and right child
            self.left_child = Tree(self.regression, self.criterion, self.n_classes)
            self.left_child._train(
                left_X,
                left_target,
                max_features,
                min_samples_split,
                max_depth - 1,
                minimum_gain,
            )

            self.right_child = Tree(self.regression, self.criterion, self.n_classes)
            self.right_child._train(
                right_X,
                right_target,
                max_features,
                min_samples_split,
                max_depth - 1,
                minimum_gain,
            )
        except AssertionError:
            self._calculate_leaf_value(target)

    def train(
        self,
        X,
        target,
        max_features=None,
        min_samples_split=10,
        max_depth=None,
        minimum_gain=0.01,
        loss=None,
    ):
        """Build a decision tree from training set.

        Parameters
        ----------

        X : array-like
            Feature dataset.
        target : dictionary or array-like
            Target values.
        max_features : int or None
            The number of features to consider when looking for the best split.
        min_samples_split : int
            The minimum number of samples required to split an internal node.
        max_depth : int
            Maximum depth of the tree.
        minimum_gain : float, default 0.01
            Minimum gain required for splitting.
        loss : function, default None
            Loss function for gradient boosting.
        """

        if not isinstance(target, dict):
            target = {"y": target}

        # Loss for gradient boosting
        if loss is not None:
            self.loss = loss

        if not self.regression:
            self.n_classes = len(np.unique(target["y"]))

        self._train(
            X,
            target,
            max_features=max_features,
            min_samples_split=min_samples_split,
            max_depth=max_depth,
            minimum_gain=minimum_gain,
        )

    def _calculate_leaf_value(self, targets):
        """Find optimal value for leaf."""
        if self.loss is not None:
            # Gradient boosting
            self.outcome = self.loss.approximate(targets["actual"], targets["y_pred"])
        else:
            # Random Forest
            if self.regression:
                # Mean value for regression task
                self.outcome = np.mean(targets["y"])
            else:
                # Probability for classification task
                self.outcome = (
                    np.bincount(targets["y"], minlength=self.n_classes)
                    / targets["y"].shape[0]
                )

    def predict_row(self, row):
        """Predict single row."""
        if not self.is_terminal:
            if row[self.column_index] < self.threshold:
                return self.left_child.predict_row(row)
            else:
                return self.right_child.predict_row(row)
        return self.outcome

    def predict(self, X):
        result = np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            result[i] = self.predict_row(X[i, :])
        return result


In [ ]:
class Loss:
    """Base class for loss functions."""

    def __init__(self, regularization=1.0):
        self.regularization = regularization

    def grad(self, actual, predicted):
        """First order gradient."""
        raise NotImplementedError()

    def hess(self, actual, predicted):
        """Second order gradient."""
        raise NotImplementedError()

    def approximate(self, actual, predicted):
        """Approximate leaf value."""
        return self.grad(actual, predicted).sum() / (
            self.hess(actual, predicted).sum() + self.regularization
        )

    def transform(self, pred):
        """Transform predictions values."""
        return pred

    def gain(self, actual, predicted):
        """Calculate gain for split search."""
        nominator = self.grad(actual, predicted).sum() ** 2
        denominator = self.hess(actual, predicted).sum() + self.regularization
        return 0.5 * (nominator / denominator)


class LeastSquaresLoss(Loss):
    """Least squares loss"""

    def grad(self, actual, predicted):
        return actual - predicted

    def hess(self, actual, predicted):
        return np.ones_like(actual)


class LogisticLoss(Loss):
    """Logistic loss."""

    def grad(self, actual, predicted):
        return actual * expit(-actual * predicted)

    def hess(self, actual, predicted):
        expits = expit(predicted)
        return expits * (1 - expits)

    def transform(self, output):
        # Apply logistic (sigmoid) function to the output
        return expit(output)


In [ ]:
class GradientBoosting(BaseEstimator):
    """Gradient boosting trees with Taylor's expansion approximation (as in xgboost)."""

    def __init__(
        self,
        n_estimators,
        learning_rate=0.1,
        max_features=10,
        max_depth=2,
        min_samples_split=10,
    ):
        self.min_samples_split = min_samples_split
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.max_features = max_features
        self.n_estimators = n_estimators
        self.trees = []
        self.loss = None

    def fit(self, X, y=None):
        self.trees = []
        self._setup_input(X, y)
        self.y_mean = np.mean(y)
        self._train()
        self.is_fitted_ = True

    def _train(self):
        # Initialize model with zeros
        y_pred = np.zeros(self.n_samples, np.float32)
        # Or mean
        # y_pred = np.full(self.n_samples, self.y_mean)

        for n in range(self.n_estimators):
            residuals = self.loss.grad(self.y, y_pred)
            tree = Tree(regression=True, criterion=mse_criterion)
            # Pass multiple target values to the tree learner
            targets = {
                # Residual values
                "y": residuals,
                # Actual target values
                "actual": self.y,
                # Predictions from previous step
                "y_pred": y_pred,
            }
            tree.train(
                self.X,
                targets,
                max_features=self.max_features,
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                loss=self.loss,
            )
            predictions = tree.predict(self.X)
            y_pred += self.learning_rate * predictions
            self.trees.append(tree)

    def _predict(self, X=None):
        y_pred = np.zeros(X.shape[0], np.float32)

        for i, tree in enumerate(self.trees):
            y_pred += self.learning_rate * tree.predict(X)
        return y_pred

    def predict(self, X=None):
        return self.loss.transform(self._predict(X))


class GradientBoostingRegressor(GradientBoosting):
    def fit(self, X, y=None):
        self.loss = LeastSquaresLoss()
        super(GradientBoostingRegressor, self).fit(X, y)


class GradientBoostingClassifier(GradientBoosting):
    def fit(self, X, y=None):
        # Convert labels from {0, 1} to {-1, 1}
        y = (y * 2) - 1
        self.loss = LogisticLoss()
        super(GradientBoostingClassifier, self).fit(X, y)

In [ ]:
class OneVsRestClassifier(BaseEstimator):

    def __init__(self, estimator):
        self.estimator = estimator
        self.estimators_ = []
        self.classes_ = []

    def fit(self, X, y=None):
        self._setup_input(X, y)
        self.classes_ = np.unique(self.y if self.y is not None else y)
        self.estimators_ = []

        for c in self.classes_:
            y_binary = (self.y == c).astype(int)
 
            clf = copy.deepcopy(self.estimator)
            
            clf.fit(self.X, y_binary)
            self.estimators_.append(clf)
            
        self.is_fitted_ = True

    def _predict(self, X=None):
        probas = np.zeros((X.shape[0], len(self.estimators_)))
        
        for i, clf in enumerate(self.estimators_):
            probas[:, i] = clf.predict(X)
            
        return probas

    def predict(self, X=None):
        if not isinstance(X, np.ndarray):
            X = np.array(X)
            
        probas = self._predict(X)
        best_indices = np.argmax(probas, axis=1)
        return self.classes_[best_indices]

## AAAA

In [ ]:
def plot_boundary_pca(model, model_args, X, y, title="title"):
    
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    
    try:
        pca = PCA(n_components=2)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_2d = pca.fit_transform(X_scaled)
    
        explained = pca.explained_variance_ratio_.sum() * 100
        print(f"PCA retains {explained:.1f}% of variance")
    
        model_2d = model(**model_args)
        model_2d.fit(X_2d, y)
    
        fig, ax = plt.subplots(figsize=(8, 6))
    
        DecisionBoundaryDisplay.from_estimator(
            model_2d,
            X_2d,
            response_method="predict",
            cmap="RdBu",
            alpha=0.4,
            ax=ax,
        )
    
        scatter = ax.scatter(
            X_2d[:, 0], X_2d[:, 1],
            c=y, cmap="RdBu",
            edgecolors="k", linewidths=0.5, s=40
        )
    
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title(title)
        plt.colorbar(scatter, ax=ax, label="Class")
        plt.tight_layout()
        plt.show()
        
    except Exception:
        print('Ignoring dataset with NA values') 

In [ ]:
def plot_metrics_CV(avg_metrics, n_folds, title=''):

    models  = list(avg_metrics.keys())
    metric_names = ["Accuracy", "F1 Score", "AUC"]
    colors  = ['mistyrose', 'salmon', 'darksalmon']

    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(8, 5))

    for i, (metric, color) in enumerate(zip(metric_names, colors)):
        means = [avg_metrics[model][metric]['mean'] for model in models]
        stds = [avg_metrics[model][metric]['std']  for model in models]
        bars = ax.barh(x + i * width, means, width, 
                       label=metric, color=color,
                       xerr=stds,
                       capsize = 4,
                       error_kw={"elinewidth": 1.2})
        for bar, mean, std in zip(bars, means, stds):
            ax.text(
                bar.get_width() + std + 0.02,
                bar.get_y() + bar.get_height() / 2,
                f"{mean:.2f}±{std:.2f}",
                ha="left", va="center", fontsize=7
            )

    ax.set_yticks(x + width)
    ax.set_yticklabels(models)
    ax.set_xlim(0, 1.1)
    ax.set_title(f"CV Metrics for {title} ({n_folds} Folds)")
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_auc(y_test, y_prob, sk_y_prob, filename):
    fig, ax = plt.subplots(figsize=(6, 6))

    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax, name="GBC")
    RocCurveDisplay.from_predictions(y_test, sk_y_prob, ax=ax, name="Sklearn GBC")
    
    ax.set_title(f"{filename} ROC Curve")

In [ ]:
def preprocess(df):

    all_nan_cols = df.columns[df.isnull().all()].tolist()
    if all_nan_cols:
        print(f"Dropping all-NaN columns: {all_nan_cols}")
        df = df.drop(columns=all_nan_cols)

    X = df.iloc[:, :-1].copy()
    y = df.iloc[:, -1].copy()

    le = LabelEncoder()
    y = le.fit_transform(y)

    categorical_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer",  SimpleImputer(strategy="most_frequent")),  
        ("encoder",  OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline,  numerical_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ])

    return preprocessor, X, y

In [ ]:
def train(df, model1, model2, n_folds):
    preprocessor, X, y = preprocess(df)

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=0)

    dataset_metrics = {
        'model1': {'Accuracy': [], 'F1 Score': [], 'AUC': []},
        'model2': {'Accuracy': [], 'F1 Score': [], 'AUC': []}
    }

    for train_index, test_index in cv.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]

        X_train = preprocessor.fit_transform(X_train)
        X_test = preprocessor.transform(X_test)

        model1.fit(X_train, y_train)
        model2.fit(X_train, y_train)

        y_prob = model1.predict(X_test)
        y_pred = (y_prob >= 0.5).astype(int)
        sk_y_pred = (model2.predict(X_test))
        sk_y_prob = (model2.predict_proba(X_test))[:, 1]
        
        for model, pred, prob in [('model1', y_pred, y_prob), ('model2', sk_y_pred, sk_y_prob)]:
            dataset_metrics[model]['Accuracy'].append(accuracy_score(y_test, pred))
            dataset_metrics[model]['F1 Score'].append(f1_score(y_test, pred))
            dataset_metrics[model]['AUC'].append(roc_auc_score(y_test, prob))

    avg_metrics = {
            model: {
                metric: {"mean": np.mean(values), "std": np.std(values)}
                for metric, values in model_metrics.items()
        } 
        for model, model_metrics in dataset_metrics.items()
    }

    return X_train, y_train, avg_metrics, dataset_metrics

In [ ]:
def compare_classifiers(model1, model2, n_folds, pca_args, dataset='', folder='', plot_pca=False, individual_metrics=False):
    overall_metrics = {
        'model1': {'Accuracy': [], 'F1 Score': [], 'AUC': []},
        'model2': {'Accuracy': [], 'F1 Score': [], 'AUC': []}
    }
    run = 1

    if folder != '':
        dir_size = len([name for name in os.listdir(folder) if name.endswith('.csv')])

        for filename in os.listdir(folder):
            if not filename.endswith('.csv'):
                continue
                
            file_path = os.path.join(folder, filename)

            print(f'[{run} | {dir_size}] Processing {filename}')
            df = pd.read_csv(file_path)

            if len(df) < n_folds:
                print(f"💀 Bypassing {filename}: Only {len(df)} samples found.")
                run += 1
                continue
                
            try:
                X_train, y_train, avg_metrics, dataset_metrics = train(df, model1, model2, n_folds)
                
                if plot_pca:
                    plot_boundary_pca(GradientBoostingClassifier, pca_args, X_train, y_train, title=filename)
                if individual_metrics:
                    plot_metrics_CV(avg_metrics, n_folds, filename) 
                
                # ⚠️ PROPERLY INDENTED: Only runs if train() succeeds! ⚠️
                for model, metrics in avg_metrics.items():
                    for metric, value in metrics.items():
                        overall_metrics[model][metric].append(value["mean"])
                        
            except Exception as e:
                # We catch EVERYTHING now. The loop WILL survive.
                print(f"💀 Skipped {filename} due to ML Error: {e}")
                
            run += 1

    else:
        df = pd.read_csv(dataset)
        try:
            X_train, y_train, avg_metrics, dataset_metrics = train(df, model1, model2, n_folds)
            if plot_pca:
                plot_boundary_pca(Sklearn_GBC, pca_args, X_train, y_train, title=dataset)
            overall_metrics = overall_metrics | dataset_metrics
        except Exception as e:
            print(f"💀 Skipped dataset due to ML Error: {e}")
    
    overall_avg_metrics = {
            model: {
                metric: {"mean": np.mean(values) if values else 0.0, "std": np.std(values) if values else 0.0}
                for metric, values in model_metrics.items()
        } 
        for model, model_metrics in overall_metrics.items()
    }

    plot_metrics_CV(overall_avg_metrics, n_folds, 'all datasets')

In [ ]:
directory = './multiclass_classification/'
dst = directory + 'dataset_1013_analcatdata_challenger.csv'

sklearn_gbc = Sklearn_GBC(n_estimators=100, learning_rate=0.1, max_depth=3, max_features=2)
gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, max_features=2)

pca = {'n_estimators':10, 'learning_rate':0.1, 'max_depth':3, 'max_features':2}

compare_classifiers(gbc, sklearn_gbc, n_folds=3, pca_args=pca, folder=directory, plot_pca=True, individual_metrics=False)

References:
https://arxiv.org/pdf/1603.02754v3.pdf

http://www.saedsayad.com/docs/xgboost.pdf

https://homes.cs.washington.edu/~tqchen/pdf/BoostedTree.pdf

http://stats.stackexchange.com/questions/202858/loss-function-approximation-with-taylor-expansion
